# Day 22 — RAG: Retrieval-Augmented Generation Concepts

**Goal:** Understand *why* RAG beats asking an LLM to "just explain"
a decision, and sketch the architecture we'll build Days 23–25.

**The problem with naive LLM explanation:**
If you ask an LLM "why was E066 assigned to Backend Dev on P001?",
it has no idea — it knows nothing about your employees, your projects,
or your optimizer's output. It would hallucinate a plausible-sounding
but completely fabricated answer.

**What RAG does instead:**
1. **Retrieve** the specific facts that justify the decision — the
   assigned employee's skills, score, the project's requirements,
   and the runner-up's details.
2. **Augment** a prompt with those retrieved facts as grounding context.
3. **Generate** a natural-language explanation using an LLM, but now
   the LLM is only summarizing facts you handed it — not inventing them.

The explanation is grounded in real data. If it's wrong, the retrieval
step is wrong — which is debuggable. A hallucinating LLM is not.

**Our LLM:** Ollama running llama3.2 locally — completely free,
no API key, no internet required after setup.

## Architecture of Our RAG Pipeline

```
staffing_plan.csv          score_matrix.csv
       │                         │
       └──────────┬──────────────┘
                  ▼
          ContextRetriever          ← src/retrieve_context.py  (Day 23)
          (pulls facts for one
           project+role assignment)
                  │
                  ▼
          Prompt Template           ← designed Day 24
          (structures facts into
           a prompt for the LLM)
                  │
                  ▼
          Ollama (llama3.2)         ← src/generate_explanation.py (Day 25)
          (free local LLM —
           summarizes grounded
           facts into plain English)
                  │
                  ▼
          Explanation Text
          (shown in dashboard)      ← src/dashboard.py (Days 26-27)
```

## What facts does our retriever need to pull?

For any assignment (project_id, role) → employee_id:

1. **Assigned employee facts:**
   - Name, actual role, experience years, availability %
   - Their profile text (built Week 2 by embed_employees.py)
   - Their match score for this slot

2. **Project facts:**
   - Project name, client, required skills for this role
   - Their profile text (built Week 2 by embed_projects.py)

3. **Runner-up facts:**
   - Who was the next-best candidate that was not chosen
   - Their score vs the winner's score (the gap)
   - Why they lost: lower score? Less available? Less experienced?

This is everything the LLM needs to write a grounded, specific,
manager-readable explanation — with no hallucination risk.

In [1]:
# Verify Ollama is running and reachable
import requests

try:
    r = requests.get("http://localhost:11434/api/tags")
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"✅ Ollama is running")
    print(f"   Available models: {models}")
except Exception as e:
    print(f"❌ Ollama not reachable: {e}")
    print("   Make sure Ollama is installed and running")

✅ Ollama is running
   Available models: ['llama3.2:latest']


## ✅ Day 22 Takeaways

- RAG = Retrieve facts → Augment prompt → Generate grounded text.
- Our retriever reads from: staffing_plan.csv, score_matrix.csv,
  employees_with_index.csv, employee_profiles.json,
  projects_with_index.csv, project_profiles.json.
- The LLM (llama3.2 via Ollama) only sees what the retriever hands it.
- Day 23: build retrieve_context.py.
- Day 24: design the prompt template.
- Day 25: wire up the Ollama call.

# Day 23 — Context Retriever

First verify the structure of our JSON files so the retriever
reads them correctly.

In [3]:
import json
import pandas as pd

# Check employee_profiles.json structure
with open(r"C:\Users\KUMAR\Desktop\staffing_copilot\data\processed\employee_profiles.json") as f:
    emp_profiles = json.load(f)

print("employee_profiles.json:")
print(f"  Type  : {type(emp_profiles)}")
print(f"  Length: {len(emp_profiles)}")
print(f"  Keys  : {list(emp_profiles[0].keys())}")
print(f"  First entry:\n  {str(emp_profiles[0])[:300]}")

employee_profiles.json:
  Type  : <class 'list'>
  Length: 80
  Keys  : ['employee_id', 'name', 'profile']
  First entry:
  {'employee_id': 'E001', 'name': 'Aryan Maharaj', 'profile': 'Frontend Dev with 1 years of experience. Expert in React. Intermediate in JavaScript, TypeScript, CSS. Beginner in GraphQL, Git, Agile/Scrum. Available at 60% capacity.'}


In [4]:
# Check project_profiles.json structure
with open(r"C:\Users\KUMAR\Desktop\staffing_copilot\data\processed\project_profiles.json") as f:
    proj_profiles = json.load(f)

print("project_profiles.json:")
print(f"  Type  : {type(proj_profiles)}")
print(f"  Length: {len(proj_profiles)}")
print(f"  Keys  : {list(proj_profiles[0].keys())}")
print(f"  First entry:\n  {str(proj_profiles[0])[:300]}")

project_profiles.json:
  Type  : <class 'list'>
  Length: 30
  Keys  : ['project_id', 'project_name', 'profile']
  First entry:
  {'project_id': 'P001', 'project_name': 'Cloud Migration — BPCL', 'profile': 'Looking for DevOps and Backend Dev and Data Engineer. Required skills: AWS, Docker, Kubernetes, Python, Terraform. Minimum 4 years experience. Deadline in 72 days. High priority high budget project.'}


In [5]:
# Check employees_with_index.csv columns
emp_df = pd.read_csv(r"C:\Users\KUMAR\Desktop\staffing_copilot\data\processed\employees_with_index.csv", index_col=0)
print("employees_with_index.csv:")
print(f"  Shape  : {emp_df.shape}")
print(f"  Columns: {list(emp_df.columns)}")
emp_df.head(3)

employees_with_index.csv:
  Shape  : (80, 10)
  Columns: ['employee_id', 'name', 'role', 'experience_years', 'skills', 'proficiency', 'availability_pct', 'cost_band', 'department', 'location']


,employee_id,name,role,experience_years,skills,proficiency,availability_pct,cost_band,department,location
0,E001,Aryan Maharaj,Frontend Dev,1,JavaScript;React;TypeScript;CSS;GraphQL;Git;Ag...,Intermediate;Expert;Intermediate;Intermediate;...,60,B,Analytics,Bangalore
1,E002,Udant Dewan,Full Stack Dev,10,JavaScript;React;Node.js;PostgreSQL;GraphQL;Do...,Expert;Expert;Expert;Expert;Intermediate;Inter...,60,B,Engineering,Hyderabad
2,E003,Gagan Sami,Backend Dev,6,Python;REST APIs;PostgreSQL;Docker;GraphQL;Git,Expert;Expert;Expert;Expert;Intermediate;Inter...,60,B,Product,Bangalore


In [6]:
# Check projects_with_index.csv columns
proj_df = pd.read_csv(r"C:\Users\KUMAR\Desktop\staffing_copilot\data\processed\projects_with_index.csv", index_col=0)
print("projects_with_index.csv:")
print(f"  Shape  : {proj_df.shape}")
print(f"  Columns: {list(proj_df.columns)}")
proj_df.head(3)

projects_with_index.csv:
  Shape  : (30, 9)
  Columns: ['project_id', 'project_name', 'client', 'required_roles', 'required_skills', 'min_experience', 'deadline_days', 'budget_band', 'priority']


,project_id,project_name,client,required_roles,required_skills,min_experience,deadline_days,budget_band,priority
0,P001,Cloud Migration — BPCL,BPCL,DevOps;Backend Dev;Data Engineer,AWS;Docker;Kubernetes;Python;Terraform,4,72,high,high
1,P002,FinTech Mobile App v3 — PhonePe,PhonePe,Backend Dev;Android Dev;Data Engineer,Python;Android;REST APIs;SQL;AWS,2,68,high,critical
2,P003,iOS Banking App — Axis Bank Tech,Axis Bank Tech,Android Dev;Backend Dev;Data Engineer,Swift;iOS;REST APIs;PostgreSQL;AWS,4,109,high,critical


In [1]:
import sys
sys.path.append("../src")
from retrieve_context import ContextRetriever

retriever = ContextRetriever("../data/processed")

all_ctx = retriever.retrieve_all()
print(f"Retrieved context for {len(all_ctx)} assignments\n")

for ctx in all_ctx:
    if "error" in ctx:
        print(f"  ❌ {ctx['error']}")
    else:
        ru_name = ctx["runner_up"]["name"] if ctx["runner_up"] else "none"
        print(f"  {ctx['project_id']} / {ctx['role']:<15} → "
              f"{ctx['assigned']['name']:<22} "
              f"score={ctx['assigned']['score']:.4f} | "
              f"runner-up: {ru_name}")

  ✅ ContextRetriever loaded: 14 assignments, 80 employees, 30 projects
Retrieved context for 14 assignments

  P001 / Backend Dev     → Urvashi Ray            score=0.8389 | runner-up: Ikbal Kothari
  P001 / Data Engineer   → Theodore Devi          score=0.7473 | runner-up: Ekavir Varkey
  P001 / DevOps          → Yashica Issac          score=0.8451 | runner-up: Raksha Varughese
  P002 / Android Dev     → Harini Choudhury       score=0.8307 | runner-up: Logan Sami
  P002 / Backend Dev     → Ikbal Kothari          score=0.8676 | runner-up: Nitesh Raghavan
  P002 / Data Engineer   → Ekavir Varkey          score=0.8118 | runner-up: Unni Bhagat
  P003 / Android Dev     → Logan Sami             score=0.7433 | runner-up: Nitesh Raghavan
  P003 / Backend Dev     → Nitesh Raghavan        score=0.8688 | runner-up: Ikbal Kothari
  P003 / Data Engineer   → Chakradev Kari         score=0.7438 | runner-up: Unni Bhagat
  P004 / Data Engineer   → Unni Bhagat            score=0.8027 | runner-up: Isaac

# Day 24 — Prompt Template Design

**Goal:** Turn the retrieved context dict into a prompt that produces
a concise, manager-readable explanation from the local LLM.

The prompt must:
- Be specific (use actual names, numbers, scores — no vague language)
- Explain why the winner was chosen and why the runner-up was not
- Stay under ~150 words output
- Instruct the model NOT to invent facts not in the context

In [2]:
def build_prompt(ctx: dict) -> str:
    assigned = ctx["assigned"]
    project  = ctx["project"]
    ru       = ctx["runner_up"]

    runner_up_section = ""
    if ru:
        direction = "higher" if ru["score_gap"] > 0 else "lower"
        runner_up_section = f"""
Runner-up considered:
- Name: {ru['name']}
- Job title: {ru['actual_role']}
- Experience: {ru['experience_years']} years
- Availability: {ru['availability_pct']}%
- Match score: {ru['score']} ({abs(ru['score_gap']):.4f} {direction} than assigned)
"""

    prompt = f"""You are a staffing coordinator writing a brief explanation for a project manager.
Explain the staffing decision below in exactly 3-4 sentences.
Use the actual names and numbers provided. Do not invent any facts not listed here.

PROJECT: {project['name']} ({ctx['project_id']})
ROLE TO FILL: {ctx['role']}
PROJECT SUMMARY: {project['summary'][:300]}

ASSIGNED CANDIDATE:
- Name: {assigned['name']}
- Job title: {assigned['actual_role']}
- Experience: {assigned['experience_years']} years
- Availability: {assigned['availability_pct']}%
- Match score: {assigned['score']} out of 1.0
- Skills: {assigned['profile'][:200]}
{runner_up_section}
Write a plain-English explanation of why {assigned['name']} was selected
for the {ctx['role']} role on {project['name']}.
If a runner-up is listed, explain in one sentence why they were not chosen."""

    return prompt


# Test: build and print the prompt for P001 Backend Dev
ctx_test = retriever.retrieve("P001", "Backend Dev")
prompt_test = build_prompt(ctx_test)
print(prompt_test)

You are a staffing coordinator writing a brief explanation for a project manager.
Explain the staffing decision below in exactly 3-4 sentences.
Use the actual names and numbers provided. Do not invent any facts not listed here.

PROJECT: Cloud Migration — BPCL (P001)
ROLE TO FILL: Backend Dev
PROJECT SUMMARY: Looking for DevOps and Backend Dev and Data Engineer. Required skills: AWS, Docker, Kubernetes, Python, Terraform. Minimum 4 years experience. Deadline in 72 days. High priority high budget project.

ASSIGNED CANDIDATE:
- Name: Urvashi Ray
- Job title: Backend Dev
- Experience: 8 years
- Availability: 100%
- Match score: 0.8389 out of 1.0
- Skills: Backend Dev with 8 years of experience. Expert in Python, REST APIs, PostgreSQL, Docker, Microservices, Git. Intermediate in Node.js, AWS. Available at 100% capacity.

Runner-up considered:
- Name: Ikbal Kothari
- Job title: Backend Dev
- Experience: 10 years
- Availability: 100%
- Match score: 0.8558 (0.0169 higher than assigned)

Writ

In [3]:
# Estimate token length — keep under ~800 to be safe with llama3.2
estimated_tokens = len(prompt_test) // 4
print(f"Prompt length  : {len(prompt_test)} characters")
print(f"Estimated tokens: ~{estimated_tokens}")
print(f"\n{'✅ Good length' if estimated_tokens < 800 else '⚠️ Consider trimming'}")

Prompt length  : 1189 characters
Estimated tokens: ~297

✅ Good length


# Day 25 — Testing the Full RAG Pipeline

In [4]:
from generate_explanation import generate_explanation

test_slots = [
    ("P001", "Backend Dev"),
    ("P002", "Android Dev"),
    ("P004", "Data Scientist"),
]

for project_id, role in test_slots:
    ctx = retriever.retrieve(project_id, role)
    explanation = generate_explanation(ctx)
    print(f"\n{'='*60}")
    print(f"📋 {ctx['project']['name']} | {role}")
    print(f"   Assigned: {ctx['assigned']['name']} "
          f"(score {ctx['assigned']['score']})")
    print(f"\n{explanation}")


📋 Cloud Migration — BPCL | Backend Dev
   Assigned: Urvashi Ray (score 0.8389)

Here's a 3-4 sentence explanation for the project manager:

Urvashi Ray was selected for the Backend Dev role on Cloud Migration - BPCL due to her extensive experience of 8 years in backend development, which aligns with the project's requirement of minimum 4 years experience. Her match score of 0.8389 out of 1.0 indicates a strong fit for the role, and her skills in Python, REST APIs, PostgreSQL, Docker, Microservices, and Git are highly relevant to the project's requirements. Additionally, her availability is at 100%, ensuring she can meet the project's deadline within 72 days. Her slightly lower match score compared to runner-up Ikbal Kothari was not enough to outweigh her other strengths and experience.

📋 FinTech Mobile App v3 — PhonePe | Android Dev
   Assigned: Harini Choudhury (score 0.8307)

Here's a 4-sentence explanation for the project manager:

Harini Choudhury was selected for the Android Dev

## ✅ RAG Pipeline Verification Checklist

For each of the 3 explanations, confirm:
- [ ] Employee's actual name appears in the text
- [ ] Explanation mentions specific experience or availability numbers
- [ ] Runner-up is mentioned with a reason for not being chosen
- [ ] No facts are invented (cross-check against raw context above)
- [ ] Length is 3-5 sentences — concise, not verbose

In [5]:
# Spot-check raw context vs explanation for P001 Backend Dev
ctx_check = retriever.retrieve("P001", "Backend Dev")
print("RAW CONTEXT — Assigned:")
for k, v in ctx_check["assigned"].items():
    if k != "profile":
        print(f"  {k}: {v}")
print("\nRAW CONTEXT — Runner-up:")
for k, v in ctx_check["runner_up"].items():
    print(f"  {k}: {v}")

RAW CONTEXT — Assigned:
  employee_id: E061
  name: Urvashi Ray
  actual_role: Backend Dev
  experience_years: 8
  availability_pct: 100
  skills: Python;REST APIs;PostgreSQL;Docker;Node.js;Microservices;AWS;Git
  score: 0.8389

RAW CONTEXT — Runner-up:
  employee_id: E066
  name: Ikbal Kothari
  actual_role: Backend Dev
  experience_years: 10
  availability_pct: 100
  skills: Python;REST APIs;PostgreSQL;Docker;AWS;System Design;Microservices;Git;Agile/Scrum
  score: 0.8558
  score_gap: 0.0169
